In [1]:
import pandas as pd
from tqdm.auto import tqdm
import sys
sys.path.append("../")
tqdm.pandas()

In [2]:
from src.repo_utils import clone_and_extract_tree

In [3]:
CONTEXT_WINDOW = 395000 # gpt 5.2's context window is 400k tokens

# TODO: change source - fix filter step

In [4]:
df_repos = pd.read_parquet("../data/02_paper_assessment/references_paper_assessment_pred.parquet.br")
# df_repos = pd.read_csv("../data/8_repo_assessment/full_dataset_LLM_pred.csv")
# df_repos = df_repos.drop_duplicates(subset="PMID", keep="first")

In [5]:
df_repos["repo_url"].value_counts()

repo_url
Appendix                                                                                            75
https://github.com/opensafely/amr-uom-brit                                                           2
https://github.com/rutgervandeleur/ecgxai                                                            2
https://github.com/pennsignals/eol-onc                                                               2
https://github.com/htx-r/Reproduce-results-from-papers                                               2
                                                                                                    ..
https://osf.io/6y8fs/                                                                                1
https://github.com/mertkarabacak/NCDB-G2G3_Glioma                                                    1
https://github.com/ohdsi-studies/lungCancerPrognostic                                                1
https://github.com/WenjuanW/Risk_Prediction_of_Post-Stroke_30-da

In [6]:
# subset rows where repo_url is present and not the literal "Appendix" (case-insensitive)
df_repos_assessment = df_repos[
    df_repos["repo_url"].notna() &
    (df_repos["is_match"] == True) &
    (df_repos["repo_url"].astype(str).str.strip().str.lower() != "appendix")
].copy()

In [ ]:
def wrapper_clone_and_extract_tree(repo_url):
    result = clone_and_extract_tree(
        repo_url=repo_url,
        context_window=CONTEXT_WINDOW,
        clone_dir="../data/03_repo_assessment/cloned_repos",
    )
    return {
        "repo_content": result.output,
        "repo_status": result.status,
        "repo_error": result.error,
    }

In [ ]:
df_repos_assessment[["repo_content", "repo_status", "repo_error"]] = (
    df_repos_assessment["repo_url"]
    .progress_apply(wrapper_clone_and_extract_tree)
    .apply(pd.Series)
)

  0%|          | 0/447 [00:00<?, ?it/s]

https://doi.org/10.11588/data/50WFVL https://heidata.uni-heidelberg.de/dataset.xhtml?persistentId=doi:10.11588/data/50WFVL Failed to resolve DOI or delegate: No specific cloner found for URL domain or format: heidata.uni-heidelberg.de
https://doi.org/10.5281/zenodo.1471616 https://zenodo.org/records/1471616 Failed to resolve DOI or delegate: [Errno 2] No such file or directory: '../data/03_repo_assessment/cloned_repos/jnkather_deep-stroma-histology_Source_code_for_deep_stroma_score_second_release/jnkather/deep-stroma-histology-v0.2.zip'
https://doi.org/10.5281/zenodo.6759730 https://zenodo.org/records/6759730 Failed to resolve DOI or delegate: [Errno 2] No such file or directory: '../data/03_repo_assessment/cloned_repos/nicorost_sparseCPM_analyses_first_revision/nicorost/sparseCPM_analyses-v1.0.1.zip'


NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead


In [ ]:
df_repos_assessment.repo_status.value_counts()

repo_status
RepoStatus.INACCESSIBLE    447
Name: count, dtype: int64